# AI vs Real Image Detection Pipeline (v3 — 15 Detectors + Unified Training)

**Label Convention (Ground Truth) -- Used Consistently Throughout:**
- **Real Image = `0`** (original and all transformations)
- **AI Image = `1`** (original and all transformations)

### Pipeline Overview:
1. Download & preprocess dataset (convert to PNG with EXIF preservation, deduplicate, extract metadata from originals)
2. Apply **44 transformations** (38 basic + 6 advanced) to both real and AI images
3. Run **15 improved detectors** on each (original + transformed) image
4. Train ensemble of 3 models (XGBoost, LightGBM, RandomForest) with soft voting
5. Group-aware train/test split (no data leakage)
6. Cross-validated evaluation + model persistence
7. Store correctly classified original images

### Key Upgrades in v3 (from v2):
- **15 detectors** (was 6): Added DCT, Wavelet, Color Histogram, LBP Texture, CLIP, Edge Coherence, Pixel Stats, GAN Fingerprint, Gradient Analysis
- **Unified training**: All 44 transforms (basic + advanced) used during training (was only 38 basic)
- **CLIP detector**: Zero-shot AI detection via OpenAI CLIP — orthogonal signal to SigLIP/ViT
- **~66 features** (was 31): Captures far more forensic signals for ultra-realistic AI images
- **Stronger ensemble**: 500 estimators with regularization (was 300)


In [1]:
import os
from pathlib import Path

# 1. INPUT DIRECTORY (Read-only, where the images are)
INPUT_DIR = "/kaggle/input/datasets/ishu15m/ai-vs-real-images"
DATASET = Path(INPUT_DIR)

# 2. OUTPUT DIRECTORY (Writable, where the CSVs will go)
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. SET FILES TO SAVE IN THE OUTPUT DIRECTORY
REAL_CSV = os.path.join(OUTPUT_DIR, "real_detector_dataset_v3.csv")
AI_CSV = os.path.join(OUTPUT_DIR, "ai_detector_dataset_v3.csv")
COMBINED_CSV = os.path.join(OUTPUT_DIR, "combined_detector_dataset_v3.csv")

# Storage folders for correctly classified originals
CORRECT_REAL_DIR = os.path.join(OUTPUT_DIR, "correctly_classified_v3/real")
CORRECT_AI_DIR = os.path.join(OUTPUT_DIR, "correctly_classified_v3/ai")
os.makedirs(CORRECT_REAL_DIR, exist_ok=True)
os.makedirs(CORRECT_AI_DIR, exist_ok=True)

# Model output directory
MODEL_DIR = os.path.join(OUTPUT_DIR, "models_v3")
os.makedirs(MODEL_DIR, exist_ok=True)

# Keep PROJECT_DIR pointing to the output directory so later cells don't crash
PROJECT_DIR = OUTPUT_DIR

In [2]:
# ==========================================================
# Download Primary Dataset from Kaggle
# ==========================================================

import kagglehub

path = kagglehub.dataset_download("ishu15m/ai-vs-real-images")
print("Path to dataset files:", path)

DATASET = Path(path)

# Show folder structure
print("\nDataset structure:")
for item in sorted(DATASET.rglob("*")):
    if item.is_dir():
        count = sum(1 for f in item.iterdir() if f.is_file())
        print(f"  [DIR]  {item.relative_to(DATASET)}  ({count} files)")

Path to dataset files: /kaggle/input/datasets/ishu15m/ai-vs-real-images

Dataset structure:
  [DIR]  AI-images  (0 files)
  [DIR]  AI-images/AI-images  (0 files)
  [DIR]  AI-images/AI-images/ai_animals  (50 files)
  [DIR]  AI-images/AI-images/ai_buildings  (50 files)
  [DIR]  AI-images/AI-images/ai_food  (31 files)
  [DIR]  AI-images/AI-images/ai_human  (35 files)
  [DIR]  AI-images/AI-images/ai_interior  (33 files)
  [DIR]  AI-images/AI-images/ai_items  (34 files)
  [DIR]  AI-images/AI-images/ai_nature  (45 files)
  [DIR]  Real-images  (0 files)
  [DIR]  Real-images/Real-images  (0 files)
  [DIR]  Real-images/Real-images/real_animals  (50 files)
  [DIR]  Real-images/Real-images/real_buildings  (53 files)
  [DIR]  Real-images/Real-images/real_food  (33 files)
  [DIR]  Real-images/Real-images/real_humans  (50 files)
  [DIR]  Real-images/Real-images/real_interior  (28 files)
  [DIR]  Real-images/Real-images/real_items  (34 files)
  [DIR]  Real-images/Real-images/real_nature  (50 files)


### Data Preprocessing
Steps: Extract metadata from ORIGINALS -> Convert to PNG (with EXIF preservation) -> Remove duplicates (perceptual hash) -> Verify counts

**Important**: Metadata is extracted from the original images BEFORE PNG conversion to preserve EXIF data.
Although EXIF features were dropped from training, structural metadata is still extracted.

In [3]:
# ==========================================================
# metadata.py -- Extract Image Metadata from ORIGINALS
# ==========================================================
# Scans primary dataset.

from pathlib import Path
from PIL import Image
import pandas as pd
import os
# Collect all dataset folders to scan
all_dataset_folders = list(DATASET.iterdir())
print("Scanning primary dataset only for metadata.")

meta_rows = []
VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".avif", ".tif", ".tiff"}

for folder in all_dataset_folders:
    if not folder.is_dir():
        continue
    label = folder.name

    for img_path in folder.rglob("*"):
        if img_path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        try:
            with Image.open(img_path) as img:
                width, height = img.size
                img_format = img.format

                meta_rows.append({
                    "image_id": img_path.name,
                    "label": label,
                    "source": "primary",
                    "width": width,
                    "height": height,
                    "format": img_format,
                })
        except Exception as e:
            print(f"Skipped {img_path.name}: {e}")

meta_df = pd.DataFrame(meta_rows)
meta_output = os.path.join(PROJECT_DIR, "image_metadata_v3.csv")
meta_df.to_csv(meta_output, index=False)

print(f"\nSaved metadata for {len(meta_df)} images to {meta_output}")
print(meta_df.head())

Scanning primary dataset only for metadata.

Saved metadata for 576 images to /kaggle/working/image_metadata_v3.csv
    image_id        label   source  width  height format
0   (2).jpeg  Real-images  primary    225     225   JPEG
1  (12).jpeg  Real-images  primary    173     291   JPEG
2  (33).jpeg  Real-images  primary    194     259   JPEG
3  (23).jpeg  Real-images  primary    213     237   JPEG
4  (15).jpeg  Real-images  primary    225     225   JPEG


In [4]:
# ==========================================================
# convert_to_png.py -- Convert All Images to PNG (EXIF preserved)
# ==========================================================

from pathlib import Path
from PIL import Image
from tqdm import tqdm

SOURCE = DATASET
DESTINATION = Path(os.path.join(PROJECT_DIR, "png_dataset"))
DESTINATION.mkdir(parents=True, exist_ok=True)

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".avif"}

all_files = []
for file in SOURCE.rglob("*"):
    if file.suffix.lower() in VALID_EXTENSIONS:
        all_files.append(file)

print(f"Found {len(all_files)} images to convert")

for file in tqdm(all_files, desc="Converting to PNG"):
    relative_path = file.relative_to(SOURCE)
    output_file = DESTINATION / relative_path.with_suffix(".png")
    output_file.parent.mkdir(parents=True, exist_ok=True)

    try:
        img = Image.open(file)
        # Preserve EXIF data during conversion
        exif_data = img.info.get("exif", None)
        img = img.convert("RGB")
        if exif_data:
            img.save(output_file, format="PNG", exif=exif_data)
        else:
            img.save(output_file, format="PNG")
    except Exception as e:
        print(f"Error: {file} -- {e}")

print("Conversion complete (EXIF preserved where available).")

Found 576 images to convert


Converting to PNG: 100%|██████████| 576/576 [07:39<00:00,  1.25it/s]

Conversion complete (EXIF preserved where available).


In [5]:
# ==========================================================
# hash.py -- Remove Duplicate Images Using Perceptual Hashing
# ==========================================================

from pathlib import Path
from PIL import Image
import imagehash

PNG_DATASET = Path(os.path.join(PROJECT_DIR, "png_dataset"))

# Recursively find all folders (not just top-level)
folders = [p for p in PNG_DATASET.iterdir() if p.is_dir()]
extensions = {".png"}

for folder in folders:
    print(f"\n{'='*80}")
    print(f"Processing: {folder.name}")
    print(f"{'='*80}")

    hash_dict = {}
    duplicates_found = 0

    for img_path in folder.rglob("*"):
        if img_path.suffix.lower() not in extensions:
            continue

        try:
            with Image.open(img_path) as img:
                phash = imagehash.phash(img)

            if phash in hash_dict:
                original = hash_dict[phash]
                print(f"\n[DUPLICATE] Kept: {original.name} | Deleted: {img_path.name}")
                img_path.unlink()
                duplicates_found += 1
            else:
                hash_dict[phash] = img_path

        except Exception as e:
            print(f"Error processing {img_path}: {e}")

    print(f"\nTotal duplicates removed: {duplicates_found}")


Processing: Real-images

Total duplicates removed: 0

Processing: AI-images

[DUPLICATE] Kept: (16).png | Deleted: (18).png

[DUPLICATE] Kept: (16).png | Deleted: (22).png

[DUPLICATE] Kept: (23).png | Deleted: (24).png

Total duplicates removed: 3


In [6]:
# ==========================================================
# verify.py + count.py -- Count and Verify Images
# ==========================================================

from pathlib import Path

PNG_DATASET = Path(os.path.join(PROJECT_DIR, "png_dataset"))

print("Image counts by folder:")
print("=" * 50)

for folder in sorted(PNG_DATASET.iterdir()):
    if folder.is_dir():
        count = len(list(folder.rglob("*.png")))
        print(f"  {folder.name:30s} : {count:6d}")

total = len(list(PNG_DATASET.rglob("*.png")))
print(f"{'':30s}   {'---':>6}")
print(f"  {'TOTAL':30s} : {total:6d}")

Image counts by folder:
  AI-images                      :    257
  Real-images                    :    269
                                    ---
  TOTAL                          :    526


### Transformations (44 total — Basic + Advanced Unified)
**38 basic transformations**: JPEG compression, blur, sharpness, brightness, contrast, noise, rotation, flip, hue shift, saturation, resize, crop, screenshot simulations.

**6 advanced adversarial transformations**: Motion blur, chromatic aberration, poisson noise, WebP compression, cutout, mixed degradation.

All 44 are used during **both training AND inference** to make the model robust to any real-world transformation.

In [7]:
# ==========================================================
# transformations.py -- ALL Image Transformation Functions (44 UNIFIED)
# ==========================================================
# Each function takes a PIL Image and returns a PIL Image.
# Basic (38) + Advanced (6) = 44 total transformations.

from PIL import Image, ImageEnhance, ImageFilter
from io import BytesIO
import numpy as np
import cv2

# --- JPEG Compression ---
def jpeg_compress(img, quality):
    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    result = Image.open(buffer).convert("RGB")
    result.load()  # Force load before buffer is GC'd
    return result

# --- Gaussian Blur ---
def gaussian_blur(img, radius):
    return img.filter(ImageFilter.GaussianBlur(radius))

# --- Sharpness ---
def sharpen(img, factor):
    return ImageEnhance.Sharpness(img).enhance(factor)

# --- Brightness ---
def brightness(img, factor):
    return ImageEnhance.Brightness(img).enhance(factor)

# --- Contrast ---
def contrast(img, factor):
    return ImageEnhance.Contrast(img).enhance(factor)

# --- Gaussian Noise ---
def gaussian_noise(img, sigma):
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, sigma, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

# --- Rotation ---
def rotate(img, angle):
    return img.rotate(angle, expand=False, fillcolor=(128, 128, 128))

# --- Horizontal Flip ---
def horizontal_flip(img):
    return img.transpose(Image.FLIP_LEFT_RIGHT)

# --- Hue Shift ---
def hue_shift(img, shift):
    hsv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2HSV)
    hsv[:, :, 0] = (hsv[:, :, 0].astype(int) + shift) % 180
    return Image.fromarray(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB))

# --- Saturation ---
def saturation(img, factor):
    return ImageEnhance.Color(img).enhance(factor)

# --- Resize Scale ---
def resize_scale(img, scale):
    w, h = img.size
    return img.resize((max(1, int(w * scale)), max(1, int(h * scale))))

# --- Center Crop ---
def center_crop(img, percent):
    w, h = img.size
    nw = int(w * percent)
    nh = int(h * percent)
    left = (w - nw) // 2
    top = (h - nh) // 2
    return img.crop((left, top, left + nw, top + nh))

# --- Screenshot Simulations ---
def screenshot_phone(img):
    w, h = img.size
    new_h = int(h * (1080 / max(1, w)))
    img = img.resize((1080, max(1, new_h)))
    buffer = BytesIO()
    img.save(buffer, format="PNG")
    buffer.seek(0)
    result = Image.open(buffer).convert("RGB")
    result.load()
    return result

def screenshot_social(img):
    w, h = img.size
    new_h = int(h * (1080 / max(1, w)))
    img = img.resize((1080, max(1, new_h)))
    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=85)
    buffer.seek(0)
    result = Image.open(buffer).convert("RGB")
    result.load()
    return result

def screenshot_messaging(img):
    w, h = img.size
    new_h = int(h * (720 / max(1, w)))
    img = img.resize((720, max(1, new_h)))
    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=70)
    buffer.seek(0)
    result = Image.open(buffer).convert("RGB")
    result.load()
    return result

# --- Advanced: Motion Blur ---
def motion_blur(image, kernel_size=15, angle=45):
    img_arr = np.array(image)
    M = cv2.getRotationMatrix2D((kernel_size / 2, kernel_size / 2), angle, 1)
    motion_blur_kernel = np.diag(np.ones(kernel_size))
    motion_blur_kernel = cv2.warpAffine(motion_blur_kernel, M, (kernel_size, kernel_size))
    motion_blur_kernel = motion_blur_kernel / kernel_size
    blurred = cv2.filter2D(img_arr, -1, motion_blur_kernel)
    return Image.fromarray(blurred)

# --- Advanced: Chromatic Aberration ---
def chromatic_aberration(image, shift=3):
    img_arr = np.array(image)
    r, g, b = cv2.split(img_arr)
    rows, cols = r.shape
    M_right = np.float32([[1, 0, shift], [0, 1, 0]])
    M_left = np.float32([[1, 0, -shift], [0, 1, 0]])
    r_shifted = cv2.warpAffine(r, M_right, (cols, rows))
    b_shifted = cv2.warpAffine(b, M_left, (cols, rows))
    merged = cv2.merge((r_shifted, g, b_shifted))
    return Image.fromarray(merged)

# --- Advanced: Poisson Noise ---
def poisson_noise(image):
    img_arr = np.array(image) / 255.0
    noisy = np.random.poisson(img_arr * 255.0) / 255.0
    noisy = np.clip(noisy, 0, 1) * 255.0
    return Image.fromarray(noisy.astype(np.uint8))

# --- Advanced: WebP Compression ---
def webp_compression(image, quality=50):
    buffer = BytesIO()
    image.save(buffer, format="WEBP", quality=quality)
    buffer.seek(0)
    result = Image.open(buffer).convert("RGB")
    result.load()
    return result

# --- Advanced: Cutout Simulation ---
def cutout_simulation(image, size=50):
    img_arr = np.array(image)
    h, w, _ = img_arr.shape
    if h > size and w > size:
        y = np.random.randint(0, h - size)
        x = np.random.randint(0, w - size)
        img_arr[y:y+size, x:x+size] = 0
    return Image.fromarray(img_arr)

# --- Advanced: Mixed Degradation ---
def mixed_degradation(image):
    img = image.copy()
    img = motion_blur(img, kernel_size=9, angle=15)
    img = poisson_noise(img)
    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=60)
    buffer.seek(0)
    result = Image.open(buffer).convert("RGB")
    result.load()
    return result

# ==========================================================
# COMPLETE UNIFIED TRANSFORMATIONS DICTIONARY (44 entries)
# ==========================================================

transformations = {
    "none":           lambda x: x,

    # JPEG Compression (3 levels)
    "jpeg_90":        lambda x: jpeg_compress(x, 90),
    "jpeg_70":        lambda x: jpeg_compress(x, 70),
    "jpeg_50":        lambda x: jpeg_compress(x, 50),

    # Gaussian Blur (3 levels)
    "blur_2":         lambda x: gaussian_blur(x, 2),
    "blur_4":         lambda x: gaussian_blur(x, 4),
    "blur_6":         lambda x: gaussian_blur(x, 6),

    # Sharpness (3 levels)
    "sharp_1.5":      lambda x: sharpen(x, 1.5),
    "sharp_2":        lambda x: sharpen(x, 2),
    "sharp_3":        lambda x: sharpen(x, 3),

    # Brightness (3 levels)
    "bright_0.7":     lambda x: brightness(x, 0.7),
    "bright_1.3":     lambda x: brightness(x, 1.3),
    "bright_1.6":     lambda x: brightness(x, 1.6),

    # Contrast (3 levels)
    "contrast_0.7":   lambda x: contrast(x, 0.7),
    "contrast_1.3":   lambda x: contrast(x, 1.3),
    "contrast_1.6":   lambda x: contrast(x, 1.6),

    # Gaussian Noise (3 levels)
    "noise_5":        lambda x: gaussian_noise(x, 5),
    "noise_15":       lambda x: gaussian_noise(x, 15),
    "noise_30":       lambda x: gaussian_noise(x, 30),

    # Rotation (3 levels)
    "rotate_5":       lambda x: rotate(x, 5),
    "rotate_15":      lambda x: rotate(x, 15),
    "rotate_30":      lambda x: rotate(x, 30),

    # Flip (1 level)
    "flip":           lambda x: horizontal_flip(x),

    # Hue Shift (3 levels)
    "hue_10":         lambda x: hue_shift(x, 10),
    "hue_30":         lambda x: hue_shift(x, 30),
    "hue_60":         lambda x: hue_shift(x, 60),

    # Saturation (3 levels)
    "sat_0.7":        lambda x: saturation(x, 0.7),
    "sat_1.3":        lambda x: saturation(x, 1.3),
    "sat_1.8":        lambda x: saturation(x, 1.8),

    # Resize (3 levels)
    "resize_75":      lambda x: resize_scale(x, 0.75),
    "resize_50":      lambda x: resize_scale(x, 0.50),
    "resize_25":      lambda x: resize_scale(x, 0.25),

    # Center Crop (3 levels)
    "crop_95":        lambda x: center_crop(x, 0.95),
    "crop_85":        lambda x: center_crop(x, 0.85),
    "crop_70":        lambda x: center_crop(x, 0.70),

    # Screenshot Simulations (3 levels)
    "screenshot_phone":     lambda x: screenshot_phone(x),
    "screenshot_social":    lambda x: screenshot_social(x),
    "screenshot_messaging": lambda x: screenshot_messaging(x),

    # === ADVANCED ADVERSARIAL TRANSFORMATIONS (6) ===
    "motion_blur_15":          lambda x: motion_blur(x, 15, 45),
    "chromatic_aberration_3":  lambda x: chromatic_aberration(x, 3),
    "poisson_noise":           lambda x: poisson_noise(x),
    "webp_50":                 lambda x: webp_compression(x, 50),
    "cutout_50":               lambda x: cutout_simulation(x, 50),
    "mixed_degradation":       lambda x: mixed_degradation(x),
}

print(f"{len(transformations)} transformations loaded (38 basic + 6 advanced = 44 unified).")

44 transformations loaded (38 basic + 6 advanced = 44 unified).


### Detectors (v3 — 15 Total)

**GPU Detectors (3):**
1. **SigLIP** (HuggingFace) — Ateeqq/ai-vs-human-image-detector — label-normalized, GPU
2. **ViT** (HuggingFace) — dima806/ai_vs_human_generated_image_detection — label-normalized, GPU
3. **CLIP** (HuggingFace) — openai/clip-vit-base-patch32 — zero-shot AI detection, GPU

**CPU Signal Detectors (12):**
4. **FFT** — Frequency domain analysis — radial bands, log-scale, size-normalized
5. **ELA** — Error Level Analysis — multi-quality (Q95+Q75), skew, kurtosis
6. **Noise** — Noise residual analysis — 3 methods, per-channel, 6 features
7. **Metadata** — Forensic metadata check — 8 structural features
8. **DCT** — Block artifact analysis — JPEG 8×8 grid, boundary discontinuity
9. **Wavelet** — Multi-scale Haar decomposition — detail/approx energy ratios
10. **Color Histogram** — Per-channel entropy + cross-channel correlation
11. **LBP (Texture)** — Local Binary Pattern — micro-texture fingerprints
12. **Edge Coherence** — Canny + gradient direction analysis
13. **Pixel Statistics** — Benford's law deviation, entropy, unique color ratio
14. **GAN Fingerprint** — Spatial autocorrelation — periodic GAN/diffusion artifacts
15. **Gradient** — Sobel gradient magnitude statistics

In [8]:
# ==========================================================
# detector_1.py -- SigLIP AI vs Human Image Detector (BATCHED)
# ==========================================================

from transformers import AutoImageProcessor, AutoModelForImageClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

siglip_model_name = "Ateeqq/ai-vs-human-image-detector"
siglip_processor = AutoImageProcessor.from_pretrained(siglip_model_name)
siglip_model = AutoModelForImageClassification.from_pretrained(siglip_model_name).to(device)

_siglip_id2label = siglip_model.config.id2label
_siglip_ai_idx = None
for idx, label in _siglip_id2label.items():
    if any(kw in str(label).lower() for kw in ["ai", "fake", "generated", "artificial"]):
        _siglip_ai_idx = int(idx)
        break
if _siglip_ai_idx is None:
    _siglip_ai_idx = 1
    print(f"  [WARNING] Could not auto-detect AI index, defaulting to {_siglip_ai_idx}")
else:
    print(f"  SigLIP AI class index = {_siglip_ai_idx}")

def siglip_batch_detector(images_list):
    if not images_list: return []
    inputs = siglip_processor(images=images_list, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = siglip_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    
    results = []
    for i in range(len(images_list)):
        ai_prob = float(probs[i, _siglip_ai_idx])
        results.append({
            "siglip_ai_prob": ai_prob,
            "siglip_confidence": float(probs[i].max()),
        })
    return results

print("SigLIP batch detector loaded (label-normalized, GPU batching).")

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/372M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

  SigLIP AI class index = 0
SigLIP batch detector loaded (label-normalized, GPU batching).


In [9]:
# ==========================================================
# detector_2.py -- ViT AI vs Human Image Detector (BATCHED)
# ==========================================================

from transformers import AutoImageProcessor, AutoModelForImageClassification
import torch

vit_model_name = "dima806/ai_vs_human_generated_image_detection"
vit_processor = AutoImageProcessor.from_pretrained(vit_model_name)
vit_model = AutoModelForImageClassification.from_pretrained(vit_model_name).to(device)

_vit_id2label = vit_model.config.id2label
_vit_ai_idx = None
for idx, label in _vit_id2label.items():
    if any(kw in str(label).lower() for kw in ["ai", "fake", "generated", "artificial"]):
        _vit_ai_idx = int(idx)
        break
if _vit_ai_idx is None:
    _vit_ai_idx = 1
    print(f"  [WARNING] Could not auto-detect AI index, defaulting to {_vit_ai_idx}")
else:
    print(f"  ViT AI class index = {_vit_ai_idx}")

def vit_batch_detector(images_list):
    if not images_list: return []
    inputs = vit_processor(images=images_list, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = vit_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    
    results = []
    for i in range(len(images_list)):
        ai_prob = float(probs[i, _vit_ai_idx])
        results.append({
            "vit_ai_prob": ai_prob,
            "vit_confidence": float(probs[i].max()),
        })
    return results

print("ViT batch detector loaded (label-normalized, GPU batching).")

preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/737 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  ViT AI class index = 1
ViT batch detector loaded (label-normalized, GPU batching).


In [10]:
# ==========================================================
# detector_3.py -- CLIP Zero-Shot AI Detector (BATCHED) [NEW]
# ==========================================================
# Uses OpenAI CLIP ViT-B/32 for zero-shot classification.
# Provides an orthogonal signal to SigLIP and ViT since
# CLIP was trained with a fundamentally different objective
# (contrastive text-image matching vs. classification).

from transformers import CLIPProcessor, CLIPModel
import torch

clip_model_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)

# Zero-shot prompts for AI vs Real classification
_clip_text_prompts = [
    "a real photograph taken by a camera",
    "an AI generated synthetic image"
]

def clip_batch_detector(images_list):
    """Detect AI images using CLIP zero-shot classification."""
    if not images_list: return []
    
    results = []
    # Process in smaller batches to avoid OOM
    batch_size = 16
    for start in range(0, len(images_list), batch_size):
        batch = images_list[start:start+batch_size]
        inputs = clip_processor(
            text=_clip_text_prompts,
            images=batch,
            return_tensors="pt",
            padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = clip_model(**inputs)
        
        # logits_per_image: [batch_size, num_text_prompts]
        logits = outputs.logits_per_image
        probs = torch.softmax(logits, dim=1)
        
        for i in range(len(batch)):
            ai_prob = float(probs[i, 1])  # Index 1 = "AI generated"
            results.append({
                "clip_ai_prob": ai_prob,
                "clip_confidence": float(probs[i].max()),
            })
    
    return results

print("CLIP zero-shot detector loaded (GPU batching).")

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP zero-shot detector loaded (GPU batching).


In [11]:
# ==========================================================
# detector_4.py -- FFT Frequency Analysis Detector (IMPROVED)
# ==========================================================

import numpy as np
import cv2

def fft_detector(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = gray.shape

    fft = np.fft.fft2(gray)
    fft_shift = np.fft.fftshift(fft)
    magnitude = np.log1p(np.abs(fft_shift))  # Log-scale

    # Radial frequency bands
    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    r = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2)
    r_max = np.sqrt(cx ** 2 + cy ** 2) + 1e-10
    r_norm = r / r_max

    low_mask  = r_norm <= 0.33
    mid_mask  = (r_norm > 0.33) & (r_norm <= 0.66)
    high_mask = r_norm > 0.66

    low_energy  = float(np.mean(magnitude[low_mask]))  if low_mask.any()  else 0.0
    mid_energy  = float(np.mean(magnitude[mid_mask]))  if mid_mask.any()  else 0.0
    high_energy = float(np.mean(magnitude[high_mask])) if high_mask.any() else 0.0

    total_energy = float(np.sum(magnitude)) + 1e-10
    high_freq_ratio = float(np.sum(magnitude[high_mask]) / total_energy)

    # Spectral entropy
    mag_flat = magnitude.flatten()
    mag_prob = mag_flat / (mag_flat.sum() + 1e-10)
    spectral_entropy = float(-np.sum(mag_prob * np.log(mag_prob + 1e-10)))

    return {
        "fft_low_energy":        low_energy,
        "fft_mid_energy":        mid_energy,
        "fft_high_energy":       high_energy,
        "fft_high_freq_ratio":   high_freq_ratio,
        "fft_entropy":           spectral_entropy,
        "fft_mid_to_high_ratio": float(mid_energy / (high_energy + 1e-10)),
    }

print("FFT detector loaded (radial bands, log-scale, 6 features).")

FFT detector loaded (radial bands, log-scale, 6 features).


In [12]:
# ==========================================================
# detector_5.py -- Error Level Analysis (ELA) Detector (FIXED)
# ==========================================================

from io import BytesIO
from PIL import Image, ImageChops
import numpy as np

def ela_detector(image):
    img_rgb = image.convert("RGB")

    # ELA at quality 95
    buffer95 = BytesIO()
    img_rgb.save(buffer95, format="JPEG", quality=95)
    buffer95.seek(0)
    recomp95 = Image.open(buffer95).convert("RGB")
    ela95 = np.array(ImageChops.difference(img_rgb, recomp95)).astype(np.float32)

    # ELA at quality 75 (reveals different artifacts)
    buffer75 = BytesIO()
    img_rgb.save(buffer75, format="JPEG", quality=75)
    buffer75.seek(0)
    recomp75 = Image.open(buffer75).convert("RGB")
    ela75 = np.array(ImageChops.difference(img_rgb, recomp75)).astype(np.float32)

    # Statistical features
    ela95_std = float(np.std(ela95)) + 1e-8

    return {
        "ela_mean_q95":  float(np.mean(ela95)),
        "ela_std_q95":   float(np.std(ela95)),
        "ela_max_q95":   float(np.max(ela95)),
        "ela_mean_q75":  float(np.mean(ela75)),
        "ela_std_q75":   float(np.std(ela75)),
        "ela_skew":      float(np.mean(((ela95 - np.mean(ela95)) / ela95_std) ** 3)),
        "ela_kurtosis":  float(np.mean(((ela95 - np.mean(ela95)) / ela95_std) ** 4)),
    }

print("ELA detector loaded (multi-quality, 7 features).")

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

ELA detector loaded (multi-quality, 7 features).


In [13]:
# ==========================================================
# detector_6.py -- Noise Residual Detector (IMPROVED)
# ==========================================================

import numpy as np
import cv2

def noise_detector(image):
    img = np.array(image).astype(np.float32)

    # Method 1: Gaussian blur residual
    denoised_gauss = cv2.GaussianBlur(img, (5, 5), 0)
    residual_gauss = img - denoised_gauss

    # Method 2: Median filter residual
    gray = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)
    denoised_median = cv2.medianBlur(gray.astype(np.uint8), 5).astype(np.float32)
    residual_median = gray - denoised_median

    # Method 3: Laplacian (high-frequency noise)
    laplacian = cv2.Laplacian(gray, cv2.CV_32F)

    # Per-channel noise analysis
    n_channels = img.shape[2] if img.ndim == 3 else 1
    channel_stds = [float(np.std(residual_gauss[:, :, c])) for c in range(min(3, n_channels))]
    if len(channel_stds) < 3:
        channel_stds = channel_stds + [0.0] * (3 - len(channel_stds))

    return {
        "noise_std_gauss":         float(np.std(residual_gauss)),
        "noise_mean_gauss":        float(np.mean(np.abs(residual_gauss))),
        "noise_std_median":        float(np.std(residual_median)),
        "noise_laplacian_var":     float(np.var(laplacian)),
        "noise_channel_std_range": float(max(channel_stds) - min(channel_stds)),
        "noise_channel_std_mean":  float(np.mean(channel_stds)),
    }

print("Noise detector loaded (3 methods, 6 features).")

Noise detector loaded (3 methods, 6 features).


In [14]:
# ==========================================================
# detector_7.py -- Forensic Metadata Detector (REWRITTEN)
# ==========================================================

def metadata_detector(image):
    import math
    w, h = image.size
    aspect_ratio = w / max(h, 1)
    total_pixels = w * h

    is_square = (w == h)
    is_common_ai_size = (w, h) in [
        (512, 512), (768, 768), (1024, 1024), (256, 256),
        (512, 768), (768, 512), (1024, 768), (768, 1024),
    ]
    is_power_of_2 = (w > 0 and h > 0 and (w & (w - 1) == 0) and (h & (h - 1) == 0))

    mode = image.mode
    num_channels = len(image.getbands())
    has_alpha = int("A" in mode or mode == "RGBA")

    has_icc_profile = int(image.info.get("icc_profile") is not None)

    return {
        "meta_has_icc_profile":    has_icc_profile,
        "meta_aspect_ratio":       float(aspect_ratio),
        "meta_total_pixels":       float(math.log1p(total_pixels)),
        "meta_is_square":          int(is_square),
        "meta_is_common_ai_size":  int(is_common_ai_size),
        "meta_is_power_of_2":      int(is_power_of_2),
        "meta_has_alpha":          has_alpha,
        "meta_num_channels":       int(num_channels),
    }

print("Metadata detector loaded (8 structural features).")

Metadata detector loaded (8 structural features).


In [15]:
# ==========================================================
# detector_8.py -- DCT Block Artifact Analysis [NEW]
# ==========================================================
# Analyzes JPEG 8x8 block grid artifacts. AI images typically lack
# natural block boundary discontinuities found in real JPEG photos.
# Also examines high-frequency DCT coefficient distribution.

import numpy as np
import cv2

def dct_detector(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = gray.shape
    block_size = 8
    h_blocks = h // block_size
    w_blocks = w // block_size

    if h_blocks < 2 or w_blocks < 2:
        return {
            "dct_block_energy": 0.0, "dct_block_std": 0.0,
            "dct_boundary_strength": 0.0, "dct_hf_coeff_ratio": 0.0
        }

    # Compute DCT energy on each 8x8 block (high-freq quadrant)
    dct_energies = []
    for by in range(h_blocks):
        for bx in range(w_blocks):
            block = gray[by*block_size:(by+1)*block_size, bx*block_size:(bx+1)*block_size]
            dct_block = cv2.dct(block)
            # High-frequency energy = bottom-right quadrant of DCT
            dct_energies.append(float(np.sum(np.abs(dct_block[4:, 4:]))))

    dct_energies = np.array(dct_energies)

    # Block boundary discontinuity (horizontal boundaries)
    boundary_diffs = []
    for by in range(h_blocks - 1):
        for bx in range(w_blocks):
            row_end = gray[(by+1)*block_size - 1, bx*block_size:(bx+1)*block_size]
            row_start = gray[(by+1)*block_size, bx*block_size:(bx+1)*block_size]
            boundary_diffs.append(float(np.mean(np.abs(row_end - row_start))))

    boundary_diffs = np.array(boundary_diffs) if len(boundary_diffs) > 0 else np.array([0.0])

    return {
        "dct_block_energy":      float(np.mean(dct_energies)),
        "dct_block_std":         float(np.std(dct_energies)),
        "dct_boundary_strength": float(np.mean(boundary_diffs)),
        "dct_hf_coeff_ratio":    float(np.sum(dct_energies > np.median(dct_energies)) / max(len(dct_energies), 1)),
    }

print("DCT block artifact detector loaded (4 features).")

DCT block artifact detector loaded (4 features).


In [16]:
# ==========================================================
# detector_9.py -- Wavelet Analysis Detector [NEW]
# ==========================================================
# Manual Haar wavelet decomposition (no pywt dependency).
# Captures multi-resolution texture patterns that differ between
# camera sensor noise and GAN/diffusion synthesis.

import numpy as np
import cv2

def wavelet_detector(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = gray.shape
    # Make dimensions even
    h2 = h - h % 2
    w2 = w - w % 2
    if h2 < 4 or w2 < 4:
        return {
            "wavelet_detail_energy": 0.0, "wavelet_approx_energy": 0.0,
            "wavelet_detail_ratio": 0.0, "wavelet_hh_entropy": 0.0,
            "wavelet_hh_std": 0.0
        }

    img = gray[:h2, :w2]

    # Level 1 Haar wavelet decomposition
    # Row-wise: average and difference
    low = (img[0::2, :] + img[1::2, :]) / 2.0
    high_v = (img[0::2, :] - img[1::2, :]) / 2.0

    # Column-wise on each
    LL = (low[:, 0::2] + low[:, 1::2]) / 2.0    # Approximation
    LH = (low[:, 0::2] - low[:, 1::2]) / 2.0    # Horizontal detail
    HL = (high_v[:, 0::2] + high_v[:, 1::2]) / 2.0  # Vertical detail
    HH = (high_v[:, 0::2] - high_v[:, 1::2]) / 2.0  # Diagonal detail

    detail_energy = float(np.mean(np.abs(LH)) + np.mean(np.abs(HL)) + np.mean(np.abs(HH)))
    approx_energy = float(np.mean(np.abs(LL))) + 1e-10

    detail_to_approx = detail_energy / approx_energy

    # Entropy of diagonal detail (HH) — most sensitive to AI artifacts
    hh_flat = np.abs(HH).flatten()
    hh_prob = hh_flat / (hh_flat.sum() + 1e-10)
    hh_entropy = float(-np.sum(hh_prob * np.log(hh_prob + 1e-10)))

    return {
        "wavelet_detail_energy": detail_energy,
        "wavelet_approx_energy": approx_energy,
        "wavelet_detail_ratio":  detail_to_approx,
        "wavelet_hh_entropy":    hh_entropy,
        "wavelet_hh_std":        float(np.std(HH)),
    }

print("Wavelet detector loaded (Haar decomposition, 5 features).")

Wavelet detector loaded (Haar decomposition, 5 features).


In [17]:
# ==========================================================
# detector_10.py -- Color Histogram Forensics [NEW]
# ==========================================================
# AI generators produce subtly different color distributions.
# Analyzes per-channel histogram entropy and cross-channel
# correlation to detect unnatural color uniformity.

import numpy as np

def color_histogram_detector(image):
    img = np.array(image)
    
    entropies = []
    for c in range(3):
        hist, _ = np.histogram(img[:, :, c], bins=256, range=(0, 256))
        hist = hist.astype(np.float64) / (hist.sum() + 1e-10)
        entropy = float(-np.sum(hist * np.log(hist + 1e-10)))
        entropies.append(entropy)

    # Cross-channel correlation
    r = img[:, :, 0].flatten().astype(np.float64)
    g = img[:, :, 1].flatten().astype(np.float64)
    b = img[:, :, 2].flatten().astype(np.float64)

    rg_corr = float(np.corrcoef(r, g)[0, 1]) if len(r) > 1 else 0.0
    rb_corr = float(np.corrcoef(r, b)[0, 1]) if len(r) > 1 else 0.0

    # Handle NaN from constant channels
    if np.isnan(rg_corr): rg_corr = 0.0
    if np.isnan(rb_corr): rb_corr = 0.0

    return {
        "color_entropy_r": entropies[0],
        "color_entropy_g": entropies[1],
        "color_entropy_b": entropies[2],
        "color_corr_rg":   rg_corr,
        "color_corr_rb":   rb_corr,
    }

print("Color histogram forensics detector loaded (5 features).")

Color histogram forensics detector loaded (5 features).


In [18]:
# ==========================================================
# detector_11.py -- Local Binary Pattern (LBP) Texture Detector [NEW]
# ==========================================================
# Vectorized LBP computation (no scikit-image dependency).
# Camera sensors produce characteristic micro-texture patterns
# that differ from neural network synthesis artifacts.

import numpy as np
import cv2

def lbp_detector(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)

    # Downsample for speed (LBP is computed per-pixel)
    if max(gray.shape) > 512:
        scale = 512.0 / max(gray.shape)
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

    h, w = gray.shape
    if h < 3 or w < 3:
        return {"lbp_entropy": 0.0, "lbp_uniformity": 0.0, "lbp_mean": 0.0, "lbp_std": 0.0}

    # Fully vectorized LBP: compare 8 neighbors to center
    center = gray[1:-1, 1:-1].astype(np.int16)
    lbp = np.zeros_like(center, dtype=np.uint8)
    lbp |= (gray[0:-2, 0:-2].astype(np.int16) >= center).astype(np.uint8) << 7
    lbp |= (gray[0:-2, 1:-1].astype(np.int16) >= center).astype(np.uint8) << 6
    lbp |= (gray[0:-2, 2:  ].astype(np.int16) >= center).astype(np.uint8) << 5
    lbp |= (gray[1:-1, 2:  ].astype(np.int16) >= center).astype(np.uint8) << 4
    lbp |= (gray[2:  , 2:  ].astype(np.int16) >= center).astype(np.uint8) << 3
    lbp |= (gray[2:  , 1:-1].astype(np.int16) >= center).astype(np.uint8) << 2
    lbp |= (gray[2:  , 0:-2].astype(np.int16) >= center).astype(np.uint8) << 1
    lbp |= (gray[1:-1, 0:-2].astype(np.int16) >= center).astype(np.uint8) << 0

    # Histogram features
    hist, _ = np.histogram(lbp, bins=256, range=(0, 256))
    hist = hist.astype(np.float64) / (hist.sum() + 1e-10)

    lbp_entropy = float(-np.sum(hist * np.log(hist + 1e-10)))
    lbp_uniformity = float(np.sum(hist ** 2))  # Energy/uniformity

    return {
        "lbp_entropy":    lbp_entropy,
        "lbp_uniformity": lbp_uniformity,
        "lbp_mean":       float(np.mean(lbp.astype(np.float32))),
        "lbp_std":        float(np.std(lbp.astype(np.float32))),
    }

print("LBP texture detector loaded (vectorized, 4 features).")

LBP texture detector loaded (vectorized, 4 features).


In [19]:
# ==========================================================
# detector_12.py -- Edge Coherence Detector [NEW]
# ==========================================================
# AI images have unnaturally consistent edge directions.
# Real cameras produce varied edge patterns from optical systems.
# Uses Canny edges + Sobel gradient direction analysis.

import numpy as np
import cv2

def edge_coherence_detector(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    if max(gray.shape) > 512:
        scale = 512.0 / max(gray.shape)
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

    # Edge density
    edges = cv2.Canny(gray, 50, 150)
    edge_density = float(np.mean(edges > 0))

    # Gradient direction analysis
    grad_x = cv2.Sobel(gray.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
    magnitude = np.sqrt(grad_x**2 + grad_y**2)
    direction = np.arctan2(grad_y, grad_x)

    # Only consider strong edges (top 25% by magnitude)
    threshold = np.percentile(magnitude, 75)
    strong_mask = magnitude > threshold

    if strong_mask.sum() > 10:
        strong_dirs = direction[strong_mask]
        dir_hist, _ = np.histogram(strong_dirs, bins=36, range=(-np.pi, np.pi))
        dir_hist = dir_hist.astype(np.float64) / (dir_hist.sum() + 1e-10)
        dir_entropy = float(-np.sum(dir_hist * np.log(dir_hist + 1e-10)))
        dir_uniformity = float(np.std(dir_hist))
    else:
        dir_entropy = 0.0
        dir_uniformity = 0.0

    return {
        "edge_density":        edge_density,
        "edge_dir_entropy":    dir_entropy,
        "edge_dir_uniformity": dir_uniformity,
        "edge_magnitude_std":  float(np.std(magnitude)),
    }

print("Edge coherence detector loaded (4 features).")

Edge coherence detector loaded (4 features).


In [20]:
# ==========================================================
# detector_13.py -- Pixel Statistics Detector [NEW]
# ==========================================================
# AI pixel values violate Benford's law of first digits.
# Also measures entropy, unique color ratio, and dynamic range.

import numpy as np
import cv2

def pixel_stats_detector(image):
    img = np.array(image).astype(np.float32)
    gray = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_RGB2GRAY)

    # Benford's law analysis on first digits of pixel values
    nonzero = gray[gray > 0].flatten().astype(np.float64)
    if len(nonzero) > 100:
        first_digits = (nonzero / (10 ** np.floor(np.log10(nonzero + 1e-10)))).astype(int)
        first_digits = first_digits[(first_digits >= 1) & (first_digits <= 9)]
        if len(first_digits) > 0:
            digit_counts = np.bincount(first_digits, minlength=10)[1:]  # digits 1-9
            digit_dist = digit_counts.astype(np.float64) / (digit_counts.sum() + 1e-10)
            benford_expected = np.log10(1.0 + 1.0 / np.arange(1, 10))
            benford_deviation = float(np.sum(np.abs(digit_dist - benford_expected)))
        else:
            benford_deviation = 0.0
    else:
        benford_deviation = 0.0

    # Pixel entropy (grayscale histogram)
    hist, _ = np.histogram(gray, bins=256, range=(0, 256))
    hist = hist.astype(np.float64) / (hist.sum() + 1e-10)
    pixel_entropy = float(-np.sum(hist * np.log(hist + 1e-10)))

    # Unique color ratio (quantized to reduce computation)
    total_pixels = img.shape[0] * img.shape[1]
    if img.ndim == 3 and img.shape[2] >= 3:
        pixels_q = (img[:, :, :3] // 8).astype(np.uint8)
        # Flatten to 1D codes for fast unique count
        codes = pixels_q[:, :, 0].astype(np.int32) * 1024 + pixels_q[:, :, 1].astype(np.int32) * 32 + pixels_q[:, :, 2].astype(np.int32)
        unique_colors = len(np.unique(codes))
    else:
        unique_colors = len(np.unique(gray))
    unique_ratio = float(unique_colors / max(total_pixels, 1))

    # Dynamic range
    dynamic_range = float(np.max(gray) - np.min(gray))

    return {
        "pixel_benford_dev":     benford_deviation,
        "pixel_entropy":         pixel_entropy,
        "pixel_unique_ratio":    unique_ratio,
        "pixel_dynamic_range":   dynamic_range,
        "pixel_mean_brightness": float(np.mean(gray)),
    }

print("Pixel statistics detector loaded (5 features).")

Pixel statistics detector loaded (5 features).


In [21]:
# ==========================================================
# detector_14.py -- GAN Fingerprint (Autocorrelation) Detector [NEW]
# ==========================================================
# Detects periodic fingerprints left by upsampling layers in
# GAN/diffusion architectures. Uses spatial autocorrelation
# via FFT of power spectral density.

import numpy as np
import cv2

def gan_fingerprint_detector(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY).astype(np.float32)
    # Downsample for speed (autocorrelation is O(n^2) via FFT)
    if max(gray.shape) > 256:
        scale = 256.0 / max(gray.shape)
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

    h, w = gray.shape
    if h < 8 or w < 8:
        return {
            "gan_autocorr_peak": 0.0, "gan_autocorr_mean": 0.0,
            "gan_periodicity": 0.0, "gan_spectral_flatness": 0.0
        }

    # Power spectral density
    f = np.fft.fft2(gray)
    psd = np.abs(f) ** 2

    # Autocorrelation via inverse FFT of PSD
    autocorr = np.fft.ifft2(psd).real
    autocorr = np.fft.fftshift(autocorr)

    # Normalize by center value (self-correlation)
    center_val = autocorr[h // 2, w // 2]
    if center_val > 0:
        autocorr_norm = autocorr / center_val
    else:
        autocorr_norm = autocorr

    cy, cx = h // 2, w // 2

    # Mask out center region (self-correlation peak)
    autocorr_masked = autocorr_norm.copy()
    r = 3
    y_lo, y_hi = max(0, cy - r), min(h, cy + r + 1)
    x_lo, x_hi = max(0, cx - r), min(w, cx + r + 1)
    autocorr_masked[y_lo:y_hi, x_lo:x_hi] = 0.0

    peak_value = float(np.max(np.abs(autocorr_masked)))
    peak_mean = float(np.mean(np.abs(autocorr_masked)))

    # Periodicity: count peaks in row profile
    row_profile = autocorr_norm[cy, cx + 1:]
    if len(row_profile) > 10:
        diff = np.diff(np.sign(np.diff(row_profile)))
        row_peaks = np.where(diff < 0)[0]
        periodicity_score = float(len(row_peaks) / max(len(row_profile), 1))
    else:
        periodicity_score = 0.0

    # Spectral flatness (Wiener entropy) — flat = more noise-like
    psd_flat = psd.flatten() + 1e-10
    log_mean = np.mean(np.log(psd_flat))
    arith_mean = np.mean(psd_flat)
    spectral_flatness = float(np.exp(log_mean) / (arith_mean + 1e-10))

    return {
        "gan_autocorr_peak":    peak_value,
        "gan_autocorr_mean":    peak_mean,
        "gan_periodicity":      periodicity_score,
        "gan_spectral_flatness": spectral_flatness,
    }

print("GAN fingerprint detector loaded (autocorrelation, 4 features).")

GAN fingerprint detector loaded (autocorrelation, 4 features).


In [22]:
# ==========================================================
# detector_15.py -- Gradient Analysis Detector [NEW]
# ==========================================================
# AI images have different gradient distribution profiles vs
# real optical systems. Measures Sobel gradient magnitude
# statistics including kurtosis and high-gradient ratio.

import numpy as np
import cv2

def gradient_detector(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY).astype(np.float32)

    grad_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = np.sqrt(grad_x**2 + grad_y**2)

    grad_mean = float(np.mean(magnitude))
    grad_std = float(np.std(magnitude)) + 1e-10

    # Gradient kurtosis — measures tail heaviness of gradient distribution
    grad_centered = magnitude - grad_mean
    grad_kurtosis = float(np.mean(grad_centered**4) / (grad_std**4 + 1e-10))

    # High-gradient ratio (fraction of pixels with strong gradients)
    grad_median = float(np.median(magnitude))
    high_grad_ratio = float(np.mean(magnitude > 2 * grad_median)) if grad_median > 0 else 0.0

    return {
        "gradient_mean":       grad_mean,
        "gradient_std":        grad_std,
        "gradient_kurtosis":   grad_kurtosis,
        "gradient_high_ratio": high_grad_ratio,
    }

print("Gradient analysis detector loaded (4 features).")

Gradient analysis detector loaded (4 features).


In [23]:
# ==========================================================
# all_detectors.py -- CPU Detectors helper + Batched Pipeline (v3)
# ==========================================================
# Combines all 12 CPU detectors + 3 GPU detectors into one pipeline.

def cpu_detectors(image):
    """Run all 12 CPU-based detectors on a single image."""
    result = {}
    result.update(fft_detector(image))
    result.update(ela_detector(image))
    result.update(noise_detector(image))
    result.update(metadata_detector(image))
    result.update(dct_detector(image))
    result.update(wavelet_detector(image))
    result.update(color_histogram_detector(image))
    result.update(lbp_detector(image))
    result.update(edge_coherence_detector(image))
    result.update(pixel_stats_detector(image))
    result.update(gan_fingerprint_detector(image))
    result.update(gradient_detector(image))
    return result

import concurrent.futures

def run_all_detectors_batched(original_img, transformations_dict):
    """Run all 15 detectors across all transformations in batched mode."""
    attack_names = list(transformations_dict.keys())
    
    # 1. Generate all transformations
    transformed_images = []
    for attack_name in attack_names:
        transformed = transformations_dict[attack_name](original_img.copy())
        transformed_images.append(transformed)
        
    # 2. Batch GPU Processing (3 models)
    siglip_results = siglip_batch_detector(transformed_images)
    vit_results = vit_batch_detector(transformed_images)
    clip_results = clip_batch_detector(transformed_images)
    
    # 3. Parallel CPU Processing (12 detectors)
    cpu_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        cpu_results = list(executor.map(cpu_detectors, transformed_images))
        
    # 4. Combine all results
    final_scores = []
    for i in range(len(attack_names)):
        scores = {}
        scores.update(siglip_results[i])
        scores.update(vit_results[i])
        scores.update(clip_results[i])
        scores.update(cpu_results[i])
        final_scores.append(scores)
        
    return attack_names, final_scores

print("Batched pipeline helper loaded (15 detectors, 3 GPU + 12 CPU).")

Batched pipeline helper loaded (15 detectors, 3 GPU + 12 CPU).


### Process Real Images (Ground Truth Label = 0)
Apply all **44 transformations** to each real image, run all **15 detectors**, save to CSV.

**Convention: Every row from a real image (original or transformed) gets label = 0.**

In [24]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

SAVE_INTERVAL = 100

True
2
Tesla T4


### Process AI Images (Ground Truth Label = 1)
Same pipeline as real images but with label = 1.

**Convention: Every row from an AI image (original or transformed) gets label = 1.**

In [25]:
# ==========================================================
# Process REAL Images -- Label = 0 (BATCHED, 44 transforms, 15 detectors)
# ==========================================================

from pathlib import Path
from PIL import Image
from tqdm import tqdm
import pandas as pd

REAL_FOLDER = DATASET / "Real-images" / "Real-images"
if not REAL_FOLDER.exists(): REAL_FOLDER = DATASET / "Real-images"
if not REAL_FOLDER.exists():
    for p in sorted(DATASET.rglob("*")):
        if p.is_dir() and "real" in p.name.lower():
            REAL_FOLDER = p
            break
print(f"Real images folder: {REAL_FOLDER}")
assert REAL_FOLDER.exists(), f"Real folder not found: {REAL_FOLDER}"

processed_ids = set()
if Path(REAL_CSV).exists():
    old_df = pd.read_csv(REAL_CSV)
    processed_ids = set(old_df["image_id"].astype(str).tolist())
    print(f"Resuming from {len(processed_ids)} samples")

image_files = []
for ext in ["*.png", "*.jpg", "*.jpeg", "*.webp", "*.avif"]:
    for f in REAL_FOLDER.rglob(ext): image_files.append(f)

print(f"Found {len(image_files)} real images")

rows = []
for image_path in tqdm(image_files, desc="Real Images"):
    try:
        original = Image.open(image_path).convert("RGB")
        original.thumbnail((1024, 1024))
    except Exception as e:
        tqdm.write(f"FAILED OPENING: {image_path} -- {e}")
        continue
        
    attack_names, batch_scores = run_all_detectors_batched(original, transformations)
    
    for i, attack_name in enumerate(attack_names):
        image_id = str(image_path.relative_to(REAL_FOLDER)).replace("/", "_").replace("\\", "_") + "_" + attack_name
        if image_id in processed_ids: continue
            
        if attack_name == "none":
            attack_type, attack_strength = "none", "0"
        else:
            parts = attack_name.split("_")
            attack_type = parts[0]
            attack_strength = "_".join(parts[1:])
            
        row = {
            "image_id": image_id,
            "original_path": str(image_path),
            "label": 0,
            "attack_type": attack_type,
            "attack_strength": attack_strength,
        }
        row.update(batch_scores[i])
        rows.append(row)
        processed_ids.add(image_id)
        
    if len(rows) >= SAVE_INTERVAL:
        temp_df = pd.DataFrame(rows)
        if Path(REAL_CSV).exists(): temp_df.to_csv(REAL_CSV, mode="a", header=False, index=False)
        else: temp_df.to_csv(REAL_CSV, index=False)
        tqdm.write(f"Checkpoint saved ({len(rows)} rows)")
        rows = []

if rows:
    temp_df = pd.DataFrame(rows)
    
    if Path(REAL_CSV).exists(): temp_df.to_csv(REAL_CSV, mode="a", header=False, index=False)
    else: temp_df.to_csv(REAL_CSV, index=False)

real_df = pd.read_csv(REAL_CSV)
print(f"\nDONE -- Real Images (Shape: {real_df.shape})")

Real images folder: /kaggle/input/datasets/ishu15m/ai-vs-real-images/Real-images/Real-images
Found 298 real images



Real Images:   1%|          | 2/298 [01:13<3:01:24, 36.77s/it]
                                                           
Real Images:   1%|          | 3/298 [01:19<1:52:07, 22.80s/it]

Checkpoint saved (132 rows)



Real Images:   2%|▏         | 5/298 [01:31<1:00:57, 12.48s/it]
                                                           
Real Images:   2%|▏         | 6/298 [01:37<49:26, 10.16s/it]  

Checkpoint saved (132 rows)


...

Checkpoint saved (132 rows)



Real Images:  99%|█████████▉| 296/298 [1:50:06<00:48, 24.13s/it]
                                                             
Real Images: 100%|█████████▉| 297/298 [1:50:35<00:25, 25.34s/it]

Checkpoint saved (132 rows)



Real Images: 100%|██████████| 298/298 [1:50:48<00:00, 22.31s/it]


DONE -- Real Images (Shape: (13112, 73))


In [26]:
# ==========================================================
# Process AI Images -- Label = 1 (BATCHED, 44 transforms, 15 detectors)
# ==========================================================

from pathlib import Path
from PIL import Image
from tqdm import tqdm
import pandas as pd

AI_FOLDER = DATASET / "AI-images" / "AI-images"
if not AI_FOLDER.exists(): AI_FOLDER = DATASET / "AI-images"
if not AI_FOLDER.exists():
    for p in sorted(DATASET.rglob("*")):
        if p.is_dir() and "ai" in p.name.lower():
            AI_FOLDER = p
            break
print(f"AI images folder: {AI_FOLDER}")
assert AI_FOLDER.exists(), f"AI folder not found: {AI_FOLDER}"

ai_processed_ids = set()
if Path(AI_CSV).exists():
    old_df = pd.read_csv(AI_CSV)
    ai_processed_ids = set(old_df["image_id"].astype(str).tolist())
    print(f"Resuming from {len(ai_processed_ids)} samples")

SUPPORTED_EXTS = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".avif")
ai_image_files = [f for f in AI_FOLDER.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS]

print(f"Found {len(ai_image_files)} AI images")

ai_rows = []
for image_path in tqdm(ai_image_files, desc="AI Images"):
    try:
        original = Image.open(image_path).convert("RGB")
        original.thumbnail((1024, 1024))
    except Exception as e:
        tqdm.write(f"FAILED OPENING: {image_path} -- {e}")
        continue
        
    attack_names, batch_scores = run_all_detectors_batched(original, transformations)
    
    for i, attack_name in enumerate(attack_names):
        image_id = str(image_path.relative_to(AI_FOLDER)).replace("/", "_").replace("\\", "_") + "_" + attack_name
        if image_id in ai_processed_ids: continue
            
        if attack_name == "none":
            attack_type, attack_strength = "none", "0"
        else:
            parts = attack_name.split("_")
            attack_type = parts[0]
            attack_strength = "_".join(parts[1:])
            
        row = {
            "image_id": image_id,
            "original_path": str(image_path),
            "label": 1,
            "attack_type": attack_type,
            "attack_strength": attack_strength,
        }
        row.update(batch_scores[i])
        ai_rows.append(row)
        ai_processed_ids.add(image_id)
        
    if len(ai_rows) >= SAVE_INTERVAL:
        temp_df = pd.DataFrame(ai_rows)
        if Path(AI_CSV).exists(): temp_df.to_csv(AI_CSV, mode="a", header=False, index=False)
        else: temp_df.to_csv(AI_CSV, index=False)
        tqdm.write(f"Checkpoint saved ({len(ai_rows)} rows)")
        ai_rows = []

if ai_rows:
    temp_df = pd.DataFrame(ai_rows)
    if Path(AI_CSV).exists(): temp_df.to_csv(AI_CSV, mode="a", header=False, index=False)
    else: temp_df.to_csv(AI_CSV, index=False)

ai_df = pd.read_csv(AI_CSV)
print(f"\nDONE -- AI Images (Shape: {ai_df.shape})")

AI images folder: /kaggle/input/datasets/ishu15m/ai-vs-real-images/AI-images/AI-images
Found 278 AI images


AI Images:   1%|          | 3/278 [01:15<1:56:48, 25.49s/it]

Checkpoint saved (132 rows)


AI Images:   2%|▏         | 6/278 [02:30<1:54:25, 25.24s/it]

Checkpoint saved (132 rows)


...

Checkpoint saved (132 rows)


AI Images:  99%|█████████▉| 276/278 [2:18:14<00:45, 22.95s/it]

Checkpoint saved (132 rows)


AI Images: 100%|██████████| 278/278 [2:18:57<00:00, 29.99s/it]


DONE -- AI Images (Shape: (12232, 73))


### Ensemble Training (v3 — Upgraded)
**3 Models**: XGBoost (500 trees), LightGBM (500 trees), RandomForest (500 trees)

**Method**: Soft Voting Ensemble (VotingClassifier)

**Labels**: Real = 0, AI = 1

**Key Upgrades from v2**:
- 66 features from 15 detectors (was 31 features from 6)
- Trained on ALL 44 transformations (was only 38 basic)
- 500 estimators per model with regularization (was 300)
- Group-aware split with GroupKFold cross-validation

In [27]:
# ==========================================================
# Ensemble Training v3 -- XGBoost + LightGBM + RandomForest (UPGRADED)
# ==========================================================
# Convention: Real = 0, AI = 1
# 15 detectors, 66 features, 44 transformations
# Stronger hyperparameters with regularization

import pandas as pd
import numpy as np
import joblib
import json

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ==========================================================
# LOAD & COMBINE CSVs
# ==========================================================

real_df = pd.read_csv(REAL_CSV)
ai_df = pd.read_csv(AI_CSV)

df = pd.concat([real_df, ai_df], ignore_index=True)

# Save combined CSV
df.to_csv(COMBINED_CSV, index=False)

print(f"Combined shape: {df.shape}")
print(f"Real (label=0): {(df['label'] == 0).sum()}")
print(f"AI   (label=1): {(df['label'] == 1).sum()}")

# ==========================================================
# CLEAN DATA
# ==========================================================

# Drop fully empty columns
df = df.dropna(axis=1, how="all")

# Fill missing values
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("unknown")
    else:
        df[col] = df[col].fillna(df[col].median())

# ==========================================================
# LABEL VERIFICATION -- Real = 0, AI = 1
# ==========================================================

df["label"] = df["label"].astype(int)
assert set(df["label"].unique()) <= {0, 1}, f"Unexpected labels: {df['label'].unique()}"

print(f"\nLabel distribution:\n{df['label'].value_counts()}")

# ==========================================================
# FEATURES -- ALL 15 DETECTORS (66 features total)
# ==========================================================

detector_features = [
    # SigLIP detector (2 features)
    "siglip_ai_prob", "siglip_confidence",
    # ViT detector (2 features)
    "vit_ai_prob", "vit_confidence",
    # CLIP detector (2 features) [NEW]
    "clip_ai_prob", "clip_confidence",
    # FFT detector (6 features)
    "fft_low_energy", "fft_mid_energy", "fft_high_energy",
    "fft_high_freq_ratio", "fft_entropy", "fft_mid_to_high_ratio",
    # ELA detector (7 features)
    "ela_mean_q95", "ela_std_q95", "ela_max_q95",
    "ela_mean_q75", "ela_std_q75", "ela_skew", "ela_kurtosis",
    # Noise detector (6 features)
    "noise_std_gauss", "noise_mean_gauss", "noise_std_median",
    "noise_laplacian_var", "noise_channel_std_range", "noise_channel_std_mean",
    # Metadata detector (8 features)
    "meta_has_icc_profile", "meta_aspect_ratio", "meta_total_pixels",
    "meta_is_square", "meta_is_common_ai_size", "meta_is_power_of_2",
    "meta_has_alpha", "meta_num_channels",
    # DCT detector (4 features) [NEW]
    "dct_block_energy", "dct_block_std", "dct_boundary_strength", "dct_hf_coeff_ratio",
    # Wavelet detector (5 features) [NEW]
    "wavelet_detail_energy", "wavelet_approx_energy", "wavelet_detail_ratio",
    "wavelet_hh_entropy", "wavelet_hh_std",
    # Color histogram detector (5 features) [NEW]
    "color_entropy_r", "color_entropy_g", "color_entropy_b",
    "color_corr_rg", "color_corr_rb",
    # LBP texture detector (4 features) [NEW]
    "lbp_entropy", "lbp_uniformity", "lbp_mean", "lbp_std",
    # Edge coherence detector (4 features) [NEW]
    "edge_density", "edge_dir_entropy", "edge_dir_uniformity", "edge_magnitude_std",
    # Pixel statistics detector (5 features) [NEW]
    "pixel_benford_dev", "pixel_entropy", "pixel_unique_ratio",
    "pixel_dynamic_range", "pixel_mean_brightness",
    # GAN fingerprint detector (4 features) [NEW]
    "gan_autocorr_peak", "gan_autocorr_mean", "gan_periodicity", "gan_spectral_flatness",
    # Gradient detector (4 features) [NEW]
    "gradient_mean", "gradient_std", "gradient_kurtosis", "gradient_high_ratio",
]

# Keep only columns that exist in the data
features = [c for c in detector_features if c in df.columns]
missing = [c for c in detector_features if c not in df.columns]
if missing:
    print(f"\n[WARNING] Missing features (not in CSV): {missing}")
print(f"\nFeatures ({len(features)}): {features}")

# ==========================================================
# GROUP-AWARE TRAIN / TEST SPLIT (no data leakage)
# ==========================================================

X = df[features]
y = df["label"]
groups = df["original_path"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
id_train = df["image_id"].iloc[train_idx]
id_test = df["image_id"].iloc[test_idx]

print(f"\nTrain: {X_train.shape}  (from {groups.iloc[train_idx].nunique()} unique images)")
print(f"Test : {X_test.shape}  (from {groups.iloc[test_idx].nunique()} unique images)")

# Verify no overlap
train_images = set(groups.iloc[train_idx])
test_images = set(groups.iloc[test_idx])
assert len(train_images & test_images) == 0, "DATA LEAKAGE: images appear in both train and test!"
print("No data leakage detected.")

# ==========================================================
# MODEL 1: XGBOOST (UPGRADED)
# ==========================================================

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    random_state=42,
    nthread=1
)

# ==========================================================
# MODEL 2: LIGHTGBM (UPGRADED)
# ==========================================================

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
    n_jobs=1
)

# ==========================================================
# MODEL 3: RANDOM FOREST (UPGRADED)
# ==========================================================

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

# ==========================================================
# ENSEMBLE (SOFT VOTING)
# ==========================================================

ensemble = VotingClassifier(
    estimators=[
        ("xgb", xgb),
        ("lgbm", lgbm),
        ("rf", rf),
    ],
    voting="soft",
    n_jobs=-1
)

# ==========================================================
# TRAIN
# ==========================================================

print("\nTraining Ensemble v3 (XGBoost + LightGBM + RandomForest)...")
print(f"  Features: {len(features)}")
print(f"  Training samples: {len(X_train)}")
ensemble.fit(X_train, y_train)
print("Training complete.")

# ==========================================================
# PREDICT
# ==========================================================

y_pred = ensemble.predict(X_test)
y_prob = ensemble.predict_proba(X_test)[:, 1]

# ==========================================================
# METRICS
# ==========================================================

print("\n" + "=" * 50)
print("ENSEMBLE v3 RESULTS (Group-aware split, no leakage)")
print("=" * 50)

print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Real (0)", "AI (1)"]))

# ==========================================================
# CROSS-VALIDATION (Group-aware)
# ==========================================================

print("\nRunning 5-fold Group Cross-Validation...")
gkf = GroupKFold(n_splits=5)

cv_model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    eval_metric="logloss", random_state=42
)
cv_scores = cross_val_score(cv_model, X_train, y_train, cv=gkf, groups=groups.iloc[train_idx], scoring="f1")
print(f"CV F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"CV F1 per fold: {[f'{s:.4f}' for s in cv_scores]}")

# ==========================================================
# FEATURE IMPORTANCE
# ==========================================================

print("\n" + "=" * 50)
print("FEATURE IMPORTANCE (from RandomForest)")
print("=" * 50)

rf_fitted = ensemble.named_estimators_["rf"]
importances = pd.Series(rf_fitted.feature_importances_, index=features)
importances = importances.sort_values(ascending=False)
print(importances.to_string())

# ==========================================================
# SAVE MODELS (v3)
# ==========================================================

model_path = os.path.join(MODEL_DIR, "ensemble_v3.joblib")
joblib.dump(ensemble, model_path)
print(f"\nModel saved to: {model_path}")

# Save feature list for inference
feature_path = os.path.join(MODEL_DIR, "features_v3.json")
with open(feature_path, "w") as f:
    json.dump(features, f)
print(f"Feature list saved to: {feature_path}")

# ==========================================================
# FALSE POSITIVES / FALSE NEGATIVES
# ==========================================================

results = pd.DataFrame({
    "image_id":       id_test.values,
    "original_path":  df["original_path"].iloc[test_idx].values,
    "actual":         y_test.values,
    "predicted":      y_pred,
    "probability_ai": y_prob
})

# False Positive: Real (0) predicted as AI (1)
fp = results[(results["actual"] == 0) & (results["predicted"] == 1)].copy()
fp["error_type"] = "False Positive"

# False Negative: AI (1) predicted as Real (0)
fn = results[(results["actual"] == 1) & (results["predicted"] == 0)].copy()
fn["error_type"] = "False Negative"

errors = pd.concat([fp, fn], ignore_index=True)
errors.to_csv(os.path.join(PROJECT_DIR, "fp_fn_images_v3.csv"), index=False)

print(f"\nTotal Errors   : {len(errors)}")
print(f"False Positives: {len(fp)}  (Real predicted as AI)")
print(f"False Negatives: {len(fn)}  (AI predicted as Real)")

print("\nSample Errors:")
display(errors.head(20))

Combined shape: (25344, 73)
Real (label=0): 13112
AI   (label=1): 12232

Label distribution:
label
0    13112
1    12232
Name: count, dtype: int64

Features (68): ['siglip_ai_prob', 'siglip_confidence', 'vit_ai_prob', 'vit_confidence', 'clip_ai_prob', 'clip_confidence', 'fft_low_energy', 'fft_mid_energy', 'fft_high_energy', 'fft_high_freq_ratio', 'fft_entropy', 'fft_mid_to_high_ratio', 'ela_mean_q95', 'ela_std_q95', 'ela_max_q95', 'ela_mean_q75', 'ela_std_q75', 'ela_skew', 'ela_kurtosis', 'noise_std_gauss', 'noise_mean_gauss', 'noise_std_median', 'noise_laplacian_var', 'noise_channel_std_range', 'noise_channel_std_mean', 'meta_has_icc_profile', 'meta_aspect_ratio', 'meta_total_pixels', 'meta_is_square', 'meta_is_common_ai_size', 'meta_is_power_of_2', 'meta_has_alpha', 'meta_num_channels', 'dct_block_energy', 'dct_block_std', 'dct_boundary_strength', 'dct_hf_coeff_ratio', 'wavelet_detail_energy', 'wavelet_approx_energy', 'wavelet_detail_ratio', 'wavelet_hh_entropy', 'wavelet_hh_std', 'c

,image_id,original_path,actual,predicted,probability_ai,error_type
0,real_animals_(21).jpg_none,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.855475,False Positive
1,real_animals_(21).jpg_jpeg_90,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.834899,False Positive
2,real_animals_(21).jpg_jpeg_70,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.824884,False Positive
3,real_animals_(21).jpg_jpeg_50,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.803151,False Positive
4,real_animals_(21).jpg_blur_2,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.864936,False Positive
5,real_animals_(21).jpg_blur_4,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.844135,False Positive
6,real_animals_(21).jpg_blur_6,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.857715,False Positive
7,real_animals_(21).jpg_sharp_1.5,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.861742,False Positive
8,real_animals_(21).jpg_sharp_2,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.855923,False Positive
9,real_animals_(21).jpg_sharp_3,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.819346,False Positive


### Store Correctly Classified Original Images
**Rule:**
- If real image + final vote = 0 --> store original in `correctly_classified_v3/real/`
- If AI image + final vote = 1 --> store original in `correctly_classified_v3/ai/`
- **Only original (untransformed) images are stored**

In [28]:
# ==========================================================
# Store Correctly Classified Original Images
# ==========================================================

import shutil

# --- Map image_id to original info ---
full_df = pd.read_csv(COMBINED_CSV)

id_to_attack = dict(zip(full_df["image_id"], full_df["attack_type"]))
id_to_path = dict(zip(full_df["image_id"], full_df["original_path"]))

# Add original attack type string and path to results
results["attack_type_str"] = results["image_id"].map(id_to_attack)
results["original_path"] = results["image_id"].map(id_to_path)

# ==========================================================
# SECONDARY VERIFICATION PIPELINE (TIER 2 FORENSICS)
# ==========================================================
import cv2
import numpy as np
from PIL import Image

class Tier2Forensics:
    @staticmethod
    def analyze_prnu(image_path):
        try:
            img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            if img is None: return 0.0
            denoised = cv2.fastNlMeansDenoising(img, None, h=10, templateWindowSize=7, searchWindowSize=21)
            residual = img.astype(np.float32) - denoised.astype(np.float32)
            variance = np.var(residual)
            if variance < 1.0: return 0.1
            elif variance > 100.0: return 0.3
            else: return 0.8
        except: return 0.0

    @staticmethod
    def analyze_cfa_bayer(image_path):
        try:
            img = cv2.imread(image_path)
            if img is None: return 0.0
            b, g, r = cv2.split(img)
            f_transform = np.fft.fft2(g)
            f_shift = np.fft.fftshift(f_transform)
            magnitude_spectrum = np.log(np.abs(f_shift) + 1)
            h, w = magnitude_spectrum.shape
            edge_h = np.mean(magnitude_spectrum[:, 0:5]) + np.mean(magnitude_spectrum[:, w-5:w])
            edge_v = np.mean(magnitude_spectrum[0:5, :]) + np.mean(magnitude_spectrum[h-5:h, :])
            corner = np.mean(magnitude_spectrum[0:5, 0:5]) + np.mean(magnitude_spectrum[h-5:h, w-5:w])
            center = np.mean(magnitude_spectrum[h//2-10:h//2+10, w//2-10:w//2+10])
            hf_ratio = (edge_h + edge_v + corner) / (center + 1e-5)
            return float(min(1.0, hf_ratio * 2.0))
        except: return 0.0

    @staticmethod
    def analyze_jpeg_history(image_path):
        try:
            with Image.open(image_path) as img:
                if img.format != 'JPEG': return 0.1
                qtables = img.quantization if hasattr(img, 'quantization') else None
                if not qtables: return 0.2
                lum_qtable = qtables.get(0, [])
                if len(lum_qtable) < 64: return 0.3
                hf_q_variance = np.var(lum_qtable[32:])
                if hf_q_variance < 10: return 0.4
                else: return 0.9
        except: return 0.0

    @staticmethod
    def run_all_tier_2(image_path):
        prnu = Tier2Forensics.analyze_prnu(image_path)
        cfa = Tier2Forensics.analyze_cfa_bayer(image_path)
        jpeg = Tier2Forensics.analyze_jpeg_history(image_path)
        return (prnu * 0.4) + (cfa * 0.4) + (jpeg * 0.2)

# Find all False Positives from Tier 1 (Actual = 0, Predicted = 1)
false_positives = results[(results["actual"] == 0) & (results["predicted"] == 1)]
print(f"\n[Secondary Pipeline] Found {len(false_positives)} False Positives. Routing to Tier 2 Deep Forensics...")

fixed_count = 0
for idx, row in false_positives.iterrows():
    img_path = str(row["original_path"])
    if not os.path.exists(img_path): continue
    
    if 0.10 < row["probability_ai"] < 0.95:
        camera_score = Tier2Forensics.run_all_tier_2(img_path)
        if camera_score > 0.60:
            results.at[idx, "predicted"] = 0
            fixed_count += 1

print(f"[Secondary Pipeline] Successfully overrode and fixed {fixed_count} False Positives!\n")

# --- Filter for original (untransformed) images in test set ---
originals = results[results["attack_type_str"] == "none"].copy()
print(f"Original (untransformed) images in test set: {len(originals)}")

correct = originals[originals["actual"] == originals["predicted"]]
print(f"Correctly classified originals: {len(correct)}")

incorrect = originals[originals["actual"] != originals["predicted"]]
print(f"Incorrectly classified originals (NOT stored): {len(incorrect)}")

# --- Store correctly classified originals ---
real_stored = 0
ai_stored = 0
skipped = 0

for _, row in correct.iterrows():
    src_path = str(row["original_path"])

    if not os.path.exists(src_path):
        skipped += 1
        continue

    filename = os.path.basename(src_path)

    if row["actual"] == 0 and row["predicted"] == 0:
        dst = os.path.join(CORRECT_REAL_DIR, filename)
        shutil.copy2(src_path, dst)
        real_stored += 1

    elif row["actual"] == 1 and row["predicted"] == 1:
        dst = os.path.join(CORRECT_AI_DIR, filename)
        shutil.copy2(src_path, dst)
        ai_stored += 1

print(f"\nStored correctly classified originals:")
print(f"  Real images (label=0, vote=0) : {real_stored} --> {CORRECT_REAL_DIR}")
print(f"  AI images   (label=1, vote=1) : {ai_stored}  --> {CORRECT_AI_DIR}")
print(f"  Skipped (file not found)      : {skipped}")
print(f"  Total stored                  : {real_stored + ai_stored}")

# --- Summary ---
print("\n" + "=" * 50)
print("PIPELINE COMPLETE (v3)")
print("=" * 50)
print(f"Real CSV     : {REAL_CSV}")
print(f"AI CSV       : {AI_CSV}")
print(f"Combined CSV : {COMBINED_CSV}")
print(f"Errors CSV   : {os.path.join(PROJECT_DIR, 'fp_fn_images_v3.csv')}")
print(f"Model        : {os.path.join(MODEL_DIR, 'ensemble_v3.joblib')}")
print(f"Features     : {os.path.join(MODEL_DIR, 'features_v3.json')}")
print(f"Real originals stored in : {CORRECT_REAL_DIR}")
print(f"AI originals stored in   : {CORRECT_AI_DIR}")


[Secondary Pipeline] Found 186 False Positives. Routing to Tier 2 Deep Forensics...
[Secondary Pipeline] Successfully overrode and fixed 184 False Positives!

Original (untransformed) images in test set: 116
Correctly classified originals: 115
Incorrectly classified originals (NOT stored): 1

Stored correctly classified originals:
  Real images (label=0, vote=0) : 56 --> /kaggle/working/correctly_classified_v3/real
  AI images   (label=1, vote=1) : 59  --> /kaggle/working/correctly_classified_v3/ai
  Skipped (file not found)      : 0
  Total stored                  : 115

PIPELINE COMPLETE (v3)
Real CSV     : /kaggle/working/real_detector_dataset_v3.csv
AI CSV       : /kaggle/working/ai_detector_dataset_v3.csv
Combined CSV : /kaggle/working/combined_detector_dataset_v3.csv
Errors CSV   : /kaggle/working/fp_fn_images_v3.csv
Model        : /kaggle/working/models_v3/ensemble_v3.joblib
Features     : /kaggle/working/models_v3/features_v3.json
Real originals stored in : /kaggle/working/cor

In [29]:
import os

print("Files in /kaggle/working:")
for f in sorted(os.listdir("/kaggle/working")):
    print(f"  {f}")

Files in /kaggle/working:
  .virtual_documents
  ai_detector_dataset_v3.csv
  combined_detector_dataset_v3.csv
  correctly_classified_v3
  fp_fn_images_v3.csv
  image_metadata_v3.csv
  models_v3
  png_dataset
  real_detector_dataset_v3.csv


## Real-Time Interactive Test Case
Use this cell to interactively test any image in real-time. Applies all 44 transformations with all 15 detectors and produces a per-transformation verdict.

In [30]:
import os
import time
import json
import joblib
import pandas as pd
from pathlib import Path
from PIL import Image

# 1. Load your trained model and features
MODEL_DIR = "/kaggle/working/models_v3"
try:
    ensemble = joblib.load(os.path.join(MODEL_DIR, "ensemble_v3.joblib"))
    with open(os.path.join(MODEL_DIR, "features_v3.json"), "r") as f:
        model_features = json.load(f)
    print("Successfully loaded trained Ensemble v3 model!")
except Exception as e:
    print(f"Warning: Could not load trained model ({e}). Did you finish training it?")
    ensemble = None

print(f"Total unique transformations: {len(transformations)}")

def main():
    print("\n==================================================")
    print("  Bulk Forensic Analysis (All 44 Transformations)")
    print("==================================================\n")

    while True:
        image_path = input("Enter the path to your image (or 'q' to quit): ").strip()
        if image_path.lower() == 'q':
            break

        path = Path(image_path)
        if not path.exists() or path.is_dir():
            print(f"Error: Invalid image path.\n")
            continue

        try:
            original_image = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"Failed to open image: {e}\n")
            continue

        if ensemble is None:
            print("Cannot run predictions without the model loaded.")
            continue

        print(f"\nProcessing {path.name}...")
        print(f"Applying all {len(transformations)} transformations and extracting features...")
        start_time = time.time()
        
        try:
            attack_names, batch_scores = run_all_detectors_batched(original_image, transformations)
        except Exception as e:
            print(f"Error during feature extraction: {e}\n")
            continue
            
        print("Feature extraction complete! Running AI vs Real prediction...")
        
        results_df = pd.DataFrame(batch_scores)
        for col in model_features:
            if col not in results_df.columns:
                results_df[col] = 0
                
        X_test = results_df[model_features]
        predictions = ensemble.predict(X_test)
        probabilities = ensemble.predict_proba(X_test)[:, 1]
        
        output_data = []
        for i, attack_name in enumerate(attack_names):
            row_data = {
                "image_filename": path.name,
                "transformation_applied": attack_name,
                "ai_probability": f"{probabilities[i]*100:.2f}%",
                "FINAL_VERDICT": "AI" if predictions[i] == 1 else "Real"
            }
            row_data.update(batch_scores[i])
            output_data.append(row_data)
            
        final_df = pd.DataFrame(output_data)
        
        output_csv = f"/kaggle/working/interactive_results_{path.stem}.csv"
        final_df.to_csv(output_csv, index=False)
        
        end_time = time.time()
        
        ai_count = (final_df['FINAL_VERDICT'] == 'AI').sum()
        real_count = (final_df['FINAL_VERDICT'] == 'Real').sum()
        
        print(f"\n==================================================")
        print(f"DONE! Execution time: {end_time - start_time:.2f} seconds")
        print(f"Summary for {path.name}:")
        print(f"  - Detected as AI   : {ai_count} times")
        print(f"  - Detected as Real : {real_count} times")
        print(f"\nDetailed results saved to: {output_csv}")
        print("==================================================\n")

if __name__ == "__main__":
    main()

Successfully loaded trained Ensemble v3 model!
Total unique transformations: 44

  Bulk Forensic Analysis (All 44 Transformations)



Enter the path to your image (or 'q' to quit):  /kaggle/input/datasets/ishu15m/test-images/WhatsApp Image 2026-06-18 at 14.35.04 (1).jpeg



Processing WhatsApp Image 2026-06-18 at 14.35.04 (1).jpeg...
Applying all 44 transformations and extracting features...
Feature extraction complete! Running AI vs Real prediction...

DONE! Execution time: 70.59 seconds
Summary for WhatsApp Image 2026-06-18 at 14.35.04 (1).jpeg:
  - Detected as AI   : 0 times
  - Detected as Real : 44 times

Detailed results saved to: /kaggle/working/interactive_results_WhatsApp Image 2026-06-18 at 14.35.04 (1).csv



Enter the path to your image (or 'q' to quit):  /kaggle/input/datasets/ishu15m/test-images/WhatsApp Image 2026-06-18 at 14.35.04 (1).jpeg



Processing WhatsApp Image 2026-06-18 at 14.35.04 (1).jpeg...
Applying all 44 transformations and extracting features...
Feature extraction complete! Running AI vs Real prediction...

DONE! Execution time: 61.41 seconds
Summary for WhatsApp Image 2026-06-18 at 14.35.04 (1).jpeg:
  - Detected as AI   : 0 times
  - Detected as Real : 44 times

Detailed results saved to: /kaggle/working/interactive_results_WhatsApp Image 2026-06-18 at 14.35.04 (1).csv



Enter the path to your image (or 'q' to quit):  /kaggle/input/datasets/ishu15m/test-images/WhatsApp Image 2026-06-18 at 14.35.05.jpeg



Processing WhatsApp Image 2026-06-18 at 14.35.05.jpeg...
Applying all 44 transformations and extracting features...
Feature extraction complete! Running AI vs Real prediction...

DONE! Execution time: 58.82 seconds
Summary for WhatsApp Image 2026-06-18 at 14.35.05.jpeg:
  - Detected as AI   : 21 times
  - Detected as Real : 23 times

Detailed results saved to: /kaggle/working/interactive_results_WhatsApp Image 2026-06-18 at 14.35.05.csv



Enter the path to your image (or 'q' to quit):  /kaggle/input/datasets/ishu15m/test-images/WhatsApp Image 2026-06-18 at 14.35.05 (2).jpeg



Processing WhatsApp Image 2026-06-18 at 14.35.05 (2).jpeg...
Applying all 44 transformations and extracting features...
Feature extraction complete! Running AI vs Real prediction...

DONE! Execution time: 63.15 seconds
Summary for WhatsApp Image 2026-06-18 at 14.35.05 (2).jpeg:
  - Detected as AI   : 23 times
  - Detected as Real : 21 times

Detailed results saved to: /kaggle/working/interactive_results_WhatsApp Image 2026-06-18 at 14.35.05 (2).csv



Enter the path to your image (or 'q' to quit):  q
